## Augmenting the Glycerolipid Dataset (GL)

This program enhances the Glycerolipid (GL) data samples using the sequence evolution and inference capabilities of ESM3.

In [1]:
import pandas as pd
import random

# Read the original CSV file
df = pd.read_csv('./dataset/positive_negative_combined/GL_Positive_Negative_70.csv', usecols=['BioDolphinID', 'Sequence', 'Label', 'protein_UniProt_ID'])

new_rows = []
for idx, row in df.iterrows():
    seq = str(row['Sequence'])
    length = len(seq)
    for i in range(2):
        # The number of masked positions: at least 10, or 30% of the sequence length
        mask_count = max(10, int(length * 0.3))
        
        # Randomly select non-repeating positions
        mask_positions = random.sample(range(length), mask_count)
        
        # Convert the sequence into a list for modification
        seq_list = list(seq)
        for pos in mask_positions:
            seq_list[pos] = '_'
        masked_seq = ''.join(seq_list)
        
        new_rows.append({
            'BioDolphinID': 'mask-' + str(row['BioDolphinID']),
            'Sequence': masked_seq,
            'Label': row['Label'],
            'protein_UniProt_ID': 'mask-' + str(row['protein_UniProt_ID'])
        })

# Create a new DataFrame and save it
df_new = pd.DataFrame(new_rows, columns=['BioDolphinID', 'Sequence', 'Label', 'protein_UniProt_ID'])
df_new.to_csv('./dataset/positive_negative_combined/Masked_GL_Positive_Negative_70.csv', index=False)


In [2]:
len(df_new)

1444

In [1]:
import pandas as pd

# Read the original data
df = pd.read_csv("./dataset/positive_negative_combined/Masked_GL_Positive_Negative_70.csv")
# Shuffle the row order
df = df.sample(frac=1, random_state=2025).reset_index(drop=True)

# Save as a new file
df.to_csv("./dataset/positive_negative_combined/Masked_GL_Positive_Negative_70.csv", index=False)

print("✅ The column order has been rearranged and the rows have been shuffled. Saved as ./dataset/positive_negative_combined/Masked_GL_Positive_Negative_70.csv")


✅ The column order has been rearranged and the rows have been shuffled. Saved as ./dataset/positive_negative_combined/Masked_GL_Positive_Negative_70.csv


In [2]:
from huggingface_hub import login
from esm.models.esm3 import ESM3
from esm.sdk.api import ESM3InferenceClient, ESMProtein, GenerationConfig
import os

# pip install --upgrade huggingface_hub

token = os.getenv("ESM_API_KEY")
login(token=token)
# This will download the model weights and instantiate the model on your machine.
model: ESM3InferenceClient = ESM3.from_pretrained("esm3-open").to("cuda") # or "cpu"


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

In [4]:
import pandas as pd
from tqdm import tqdm

# Example function (slightly completed based on your provided structure)
def GenerateSeq(BioDolphinIDs, Sequences, Labels, protein_UniProt_IDs):
    new_seq = []
    for seq in tqdm(Sequences, desc="Generating sequences"):
        protein = ESMProtein(sequence=seq)
        # Assume that the model and GenerationConfig have been properly initialized
        protein = model.generate(
            protein,
            GenerationConfig(track="sequence", num_steps=24, temperature=0.0)
        )
        new_seq.append(protein.sequence)
        break
    return new_seq

# 1. Read the original data
df = pd.read_csv('./dataset/positive_negative_combined/Masked_GL_Positive_Negative_70.csv', usecols=['BioDolphinID', 'Sequence', 'Label', 'protein_UniProt_ID'])

# 2. Pass data into the function
BioDolphinIDs = df['BioDolphinID'].tolist()
Sequences = df['Sequence'].tolist()
Labels = df['Label'].tolist()
protein_UniProt_IDs = df['protein_UniProt_ID'].tolist()

# 3. Generate new sequences
Generated_Sequences = GenerateSeq(BioDolphinIDs, Sequences, Labels, protein_UniProt_IDs)

# 4. Build the result DataFrame
df_result = pd.DataFrame({
    'BioDolphinID': BioDolphinIDs,
    'Sequence': Generated_Sequences,
    'Label': Labels,
    'protein_UniProt_ID': protein_UniProt_IDs
})

# 5. Save as a new file
df_result.to_csv('./dataset/positive_negative_combined/finished_Masked_GL_Positive_Negative_70.csv', index=False)


In [ ]:
import pandas as pd

# List of files to be merged
files = [
    './dataset/positive_negative_combined/finished_Masked_GL_Positive_Negative_70.csv',
    './dataset/positive_negneative_combid/GL_Positive_Negative_70.csv',
]

# Read and merge all files
df_list = [pd.read_csv(file) for file in files]
df_merged = pd.concat(df_list, ignore_index=True)

# Shuffle the row order
df_merged = df_merged.sample(frac=1, random_state=2025).reset_index(drop=True)

# Save as a new CSV file
df_merged.to_csv('./dataset/positive_negneative_combid/New_mask_30_2_GL_Positive_Negative_70.csv', index=False)


In [ ]:
len(df_merged)